# SFINCS Demo — NJ Sandy (Phase 3: quadtree + SnapWave + IG)

Quadtree variable-resolution rebuild (200 → 100 → 50 → 25 m, refined toward the dune line) of the working regular-grid Sandy notebook. Adds the SnapWave incident wave solver with IG-band binding, replacing the parametric Stockdon water-level hack with a real spatially-varying wave field.


## Overview & modeling choices

A **hindcast of Hurricane Sandy (28–31 Oct 2012)** flooding on the NJ coast from Sandy Hook to Asbury Park, in three phases: **Phase 1** (slow, one-time geometry build), **Phase 2** (fast forcing + run), **Phase 3** (flood maps + validation).

### Modeling choices at a glance

| Aspect | Choice |
|---|---|
| Event / window | Hurricane Sandy, 2012-10-28 → 10-31 UTC (landfall ≈ 10-29 23:30) |
| Domain | NJ coast, Sandy Hook → Asbury Park (~40.15–40.50 °N) |
| Grid | 50 m rotated UTM 18N, 8 subgrid pixels (~3 m effective) |
| Elevation | 4-tier merge, **pre-Sandy 2010 USACE NCMP on top** (captures the pre-replenishment dune state) |
| Active / outflow | `zb ≥ −10 m` active; `mask=3` outflow on lateral lows |
| Water-level boundary | NOAA CO-OPS gauges (Battery, Atlantic City, Cape May), `buffer=100 km`, 2 alongshore support points. Sandy Hook excluded (failed mid-storm) |
| Wave setup | **Stockdon (2006)** parametric at the boundary, `β_f=0.05`, from ERA5 waves per support point — adds setup, not runup |
| Wind + pressure | ERA5 hourly |
| Rainfall | NOAA AORC v1.1 (~1 km hourly), ~34 mm here |
| Discharge | USGS daily-mean (Shark R, Navesink) at the in-domain estuary inflow cells |
| Infiltration | SCS Curve Number (NLCD × SSURGO HSG) — consumes rainfall only |
| Coriolis / advection | both on (lat 40.32°) |
| Solver | SFINCS v2.3.2 (Docker), regular grid + subgrid |
| Validation | Sandy Hook gauge + USGS pre-storm gauges + 1 storm-tide sensor + **31 USGS HWMs** + **FEMA MOTF** extent |

**Next steps on the roadmap:** SnapWave + IG wavemakers on a quadtree rebuild (the real fix for open-coast runup).

## Data Sources

**Elevation / topobathy** (merged top → bottom; later entries only fill NoData):

| Dataset | File | Purpose |
|---|---|---|
| **2010 USACE NCMP topobathy** (NOAA ID 9456) | `data/elevation/usace_nj_2010_topobathy.tif` | 1 m **pre-Sandy** topobathy on dune/beach/nearshore. NAVD88. |
| NOAA CUDEM 1/9″ | `data/elevation/cudem_asbury.tif` | ~3 m fill for inlets + shelf where NCMP has no data. NAVD88. |
| NJ 10-ft LiDAR | `data/elevation/nj_10ft_dem.tif` | Inland (`zmin=0.001` excludes hydroflattened water). NAVD88. |
| GEBCO 2026 | `data/elevation/gebco_nj.tif` | Offshore tail (≥ ~450 m). |

> *Why pre-Sandy data?* Modern DEMs include post-storm beach replenishment and engineered dunes — using them systematically under-predicts overtopping.

**Forcing:**

| Dataset | File | Purpose |
|---|---|---|
| NOAA CO-OPS water levels (4 gauges) | `data/gtsm/noaa_sandy_nj.nc` | Hourly NAVD88; Sandy Hook excluded (failed mid-storm) and lives in the validation file. |
| ERA5 winds + MSLP | `data/era5/era5_nj_sandy_2012_10_28_31.nc` | Hourly, Oct 28–31 2012. |
| ERA5 wave field | `data/waves/era5_waves_nj.nc` | Hs, Tp per boundary support point for the Stockdon setup. |
| NOAA AORC rainfall | `data/precip/aorc_sandy_nj.nc` | ~1 km hourly QPE (covers 2012 unlike MRMS). |
| USGS river discharge | `data/discharge/usgs_sandy_discharge.nc` | Daily-mean for Shark R + Navesink/Shrewsbury. |

**Roughness + infiltration:**

| Dataset | File | Purpose |
|---|---|---|
| NLCD 2012 | `data/roughness/nlcd_2012.tif` | Manning's-n reclass via `data/roughness/NLCD_CONUS_mapping.csv` (Bunya/Atkinson Atlantic-coast values, NJ-tuned at classes 23/24) + the LULC half of CN. |
| SCS Curve Number | `data/infiltration/cn_nj.nc` | NLCD × SSURGO HSG → infiltration grid. |

**Validation:**

| Dataset | File | Use |
|---|---|---|
| NOAA Sandy Hook (pre-failure) | `data/gtsm/noaa_sandy_validation.nc` | Temporal check up to gauge failure. |
| USGS in-domain tidal gauges | `data/gtsm/usgs_sandy_tidal_nj.nc` | Pre-storm tide range/phase at Shark R + Shrewsbury. |
| USGS storm-tide sensor | `data/gtsm/sandy_storm_tide_nj.nc` | The one in-domain record that survived the peak (open coast 40.37). |
| USGS HWMs (31, in-domain) | `data/validation/sandy_hwms.geojson` | Spatial peak validation. |
| FEMA MOTF surge extent | `data/validation/sandy_motf_extent.tif` | Spatial flood-extent consistency (CSI/POD/FAR). |

In [1]:
# Imports
import os
import subprocess
from datetime import datetime
from pathlib import Path

import geopandas as gpd
import hvplot.pandas
import hvplot.xarray
import matplotlib.pyplot as plt
import pandas as pd
import rioxarray
import xarray as xr
from hydromt._utils import log
from hydromt_sfincs import DATADIR, SfincsModel, utils
from shapely.geometry import Point


## Phase 1 — Static build

Everything here depends only on **geometry, topography, and roughness** — grid, elevation, mask, observation points, and the subgrid tables. It's the expensive part (the subgrid build spikes to ~13 GB RAM), but it's a **one-time cost**: the outputs are written to disk and SFINCS just reads them.

**Re-run Phase 1 only when one of these changes:** region/resolution, the elevation datasets or merge order, the roughness datasets or reclass table, `nr_subgrid_pixels`, or the mask `zmin`/boundary logic.

If you're only changing **forcing** (water level, wind, pressure, timing), skip straight to Phase 2 — it reopens this model from disk and never touches the subgrid build.

### 1. Initialize SfincsModel, set data library, and output folder

In [2]:
model_root = "../model_quadtree"

log.initialize_logging()
log.set_log_level(log_level=30)  # Only errors and critical

# Add file handler to log to a file
log_file = Path(model_root) / "hydromt_sfincs.log"
Path(model_root).mkdir(parents=True, exist_ok=True)
logger = log._add_filehandler(log_file)

# Add data catalogs
data_libs = ["/home/zagreus/nj_sandy_sfincs/data/data_catalog.yml"]

# Create model. write_gis=True dumps gis/*.tif/geojson for QGIS inspection
# (same as the regular-grid notebook).
sf = SfincsModel(data_libs=data_libs, root=model_root, mode="w+", write_gis=True)


2026-06-02 11:04:06,204 - hydromt.model.model - model - WARNING - No region component found in components.


### 2. Specify grid

In [3]:
# Phase 3: QUADTREE grid (replaces the 50 m regular grid).
# Base resolution 200 m; refinement polygons step it down to 100 / 50 / 25 m
# nested toward the surf zone. See scripts/build_quadtree_refinement.py for
# the polygon recipe (full region @ L1, coastal corridor @ L2, dune+surf @ L3
# gated by elevation). Cells outside any polygon stay at 200 m.
refinement_gdf = gpd.read_file(
    "/home/zagreus/nj_sandy_sfincs/data/quadtree/refinement_polygons.geojson"
)

# elevation_list is passed here so the level-2/3 polygons can use their
# zmin/zmax columns to gate refinement by topobathy (quadtree_builder.py
# only applies elevation filtering when elevation_list is non-empty).
elevation_list = [
    {"elevation": "usace_nj_2010"},  # 1 m pre-Sandy topobathy
    {"elevation": "cudem_nj"},  # 3 m fill: inlet + shelf
    {"elevation": "nj_10ft_dem", "zmin": 0.001},  # 3 m fill: inland
    {"elevation": "gebco_nj"},  # 450 m offshore tail
]

sf.quadtree_grid.create_from_region(
    region={"geom": "/home/zagreus/nj_sandy_sfincs/data/region.geojson"},
    res=200,  # base level-0 cell size
    rotated=True,
    crs="utm",
    refinement_polygons=refinement_gdf,
    elevation_list=elevation_list,
)

# Quick cell-count sanity check. Aborts if the grid blew past our 24 GB
# headroom — the subgrid build is the memory peak; conservatively budget
# ~500k cells before worrying.
qg = sf.quadtree_grid.data
nlev = int(qg.attrs["nr_levels"])
n_total = int(qg.grid.n_face)
print(f"Quadtree: {n_total} cells across {nlev} refinement levels")
for ilev in range(nlev):
    nlev_cells = int((qg["level"] == ilev).sum())
    dx_lev = qg.attrs["dx"] / (2**ilev)
    print(f"  level {ilev}: {nlev_cells:>7d} cells @ ~{dx_lev:.0f} m")
assert n_total < 500_000, (
    f"Quadtree has {n_total} cells — too many for 24 GB RAM. Tune refinement."
)


Refining ...
Time elapsed : 30.075512409210205 s
Finding neighbors ...
Time elapsed : 0.037252187728881836 s
Setting neighbors left and below ...
Time elapsed : 0.48956799507141113 s
Getting uv points ...
Time elapsed : 0.13289475440979004 s
Making XUGrid ...
Got rid of duplicates in 0.5348 seconds
Made XUGrid in 0.0025 seconds
Quadtree: 339174 cells across 4 refinement levels
  level 0:       0 cells @ ~200 m
  level 1:      46 cells @ ~100 m
  level 2:   22478 cells @ ~50 m
  level 3:  131346 cells @ ~25 m


### 3. Add elevation

In [ ]:
# Elevation onto the quadtree mesh. Same 4-tier merge as the regular-grid
# notebook; quadtree_elevation.create samples each refinement level at its
# native resolution (level 3 = 25 m, samples USACE 1 m densely; level 0 =
# 200 m, samples GEBCO). nrmax=200 chunks the reproject — memory-friendly.
elevation_list = [
    {"elevation": "usace_nj_2010"},  # 1 m pre-Sandy topobathy
    {"elevation": "cudem_nj"},  # 3 m fill: inlet + shelf
    {"elevation": "nj_10ft_dem", "zmin": 0.001},  # 3 m fill: inland
    {"elevation": "gebco_nj"},  # 450 m offshore tail
]

sf.quadtree_elevation.create(elevation_list=elevation_list, buffer_cells=0, nrmax=200)

print(
    f"z range: {float(sf.quadtree_grid.data['z'].min()):.1f} .. "
    f"{float(sf.quadtree_grid.data['z'].max()):.1f} m"
)


### 4. Make mask of activate and inactive cells

In [ ]:
# Active mask on the quadtree grid. Same -10 m criterion as the regular grid:
# cells with z >= -10 m are active (NJ shelf is shallow enough to capture).
sf.quadtree_mask.create_active(zmin=-10)

print(
    f"active sfincs cells: {int((sf.quadtree_grid.data['mask'] > 0).sum())} "
    f"/ {sf.quadtree_grid.data.grid.n_face}"
)


: 

### 5. Update mask with boundary cells

Two boundary types are set on the perimeter of the active mask:
- **Water level (mask=2)** on the deep-ocean edge (`zmax=-1`) — where NOAA CO-OPS forcing is applied.
- **Outflow (mask=3)** on the lateral/inland edges at low elevation (`-1 ≤ zb ≤ 2 m`) — lets surge that propagates into back-bay channels (Shark River, Sandy Hook Bay) drain out of the domain instead of piling up against the model edge.

In [ ]:
# Boundary cells on the quadtree grid (same rules as the regular grid):
#  - mask=2: offshore water-level boundary where z <= -1 m (reset_bounds=True).
#  - mask=3: lateral/inland outflow on channel/back-bay edges; reset_bounds=
#    False keeps the mask=2 cells just set.
sf.quadtree_mask.create_boundary(btype="waterlevel", zmax=-1, reset_bounds=True)
sf.quadtree_mask.create_boundary(btype="outflow", zmin=-1, zmax=2, reset_bounds=False)

m = sf.quadtree_grid.data["mask"]
print(f"waterlevel boundary (mask=2): {int((m == 2).sum())}")
print(f"outflow boundary    (mask=3): {int((m == 3).sum())}")

: 

### 5b. SnapWave mask = SFINCS mask (X1 setup)

SnapWave needs a `snapwave_mask` on the quadtree grid. Earlier configs gave it a **wider** mask (active to −50 m, a wave boundary at z ≤ −15 m) plus a wavemaker line, following Leijnse — but that **deterministically crashed SFINCS**. Root cause (2026-05-26): `hydromt`'s `create_from_grid` placed the wave input points at **raw ERA5 grid coordinates, outside the model mesh**, where SnapWave read depth = 0 m, clamped to 5 m, and the IG-bound `Hm0` exploded.

The working **X1** setup sets `snapwave_mask = SFINCS mask` and injects the incident spectrum at the offshore SFINCS **waterlevel boundary** (`mask==2`, ≈ −10 m) — points guaranteed inside the mesh in real water. Wave setup is handed to SFINCS across the shared active domain (no wavemaker line). See memory `project-snapwave-root-cause` and `project-hm0-spike-rootcause`.

In [ ]:
import numpy as np

# X1 SnapWave mask = SFINCS mask exactly. create_active() instantiates the
# snapwave_mask variable on the grid; we then overwrite it with the SFINCS mask
# so the wave solver and the hydrodynamic solver share one domain. The incident
# spectrum is injected at the offshore SFINCS waterlevel boundary (mask==2);
# no separate wider SnapWave mask and no wavemaker line (those crashed -- see
# the markdown above and memory project-snapwave-root-cause).
sf.quadtree_snapwave_mask.create_active(zmin=-10)
_sw = sf.quadtree_grid.data["snapwave_mask"]
sf.quadtree_grid.data["snapwave_mask"] = _sw.copy(
    data=sf.quadtree_grid.data["mask"].values.copy()
)
_sw = sf.quadtree_grid.data["snapwave_mask"]
print(
    "snapwave_mask = SFINCS mask:",
    {int(k): int(v) for k, v in zip(*np.unique(_sw.values, return_counts=True))},
)

: 

### 6. Add observation points

In [ ]:
# Project obs points + the Sandy Hook NOAA gauge for validation. The Sandy
# Hook 8531680 gauge falls inside the domain — modeled zs(t) here vs the
# observed record (valid up to the gauge failure ~23:00 UTC 2012-10-29) is the
# cleanest single-point temporal check; the USGS HWMs (Phase 3) add the
# spatial check across the domain.
obs_gdf = gpd.read_file("../data/obs.geojson")
sandy_hook = gpd.GeoDataFrame(
    {"name": ["sandy_hook_gauge"]},
    geometry=[Point(-74.0091, 40.4669)],  # NOAA 8531680
    crs="EPSG:4326",
)
obs_all = gpd.GeoDataFrame(
    pd.concat([obs_gdf, sandy_hook], ignore_index=True),
    crs="EPSG:4326",
)
sf.observation_points.create(locations=obs_all, merge=False)

_ = sf.plot_basemap(
    variable="dep",
    plot_geoms=True,
    plot_bounds=True,
    bmap="sat",
    zoomlevel=12,
)

: 

### 7. Make subgrid derived tables

Subgrid tables bake elevation and roughness into per-cell lookup tables at higher effective resolution. SFINCS uses these directly and **ignores any standalone manningfile** when subgrid is active, so we set roughness here rather than as a separate model layer.

In [ ]:
# Drop the data-catalog cache from the elevation step before reloading the
# DEMs for subgrid. hydromt keeps every RasterDataset it has read alive on
# the catalog object — without this, the cached USACE 1 m topobathy etc.
# stay resident through the subgrid build and double up the peak memory.
import gc

for src in list(sf.data_catalog.sources):
    s = sf.data_catalog.get_source(src)
    if hasattr(s, "_data"):
        s._data = None
gc.collect()

# Roughness on the quadtree mesh, then the subgrid V-h tables.
#
# quadtree_roughness.create writes a per-cell `manning` from the NLCD reclass
# (same NJ-tuned table as the regular grid — Bunya/Atkinson Atlantic-coast
# values, classes 23/24 nudged to 0.10 / 0.13). quadtree_subgrid.create then
# samples each cell at nr_subgrid_pixels x nr_subgrid_pixels finer points to
# build the volume / conveyance / roughness lookup tables.
#
# Effective subgrid resolution per cell level (with nr_subgrid_pixels=8):
#   level 0 (200 m)  -> 25 m subgrid sampling
#   level 1 (100 m)  -> 12.5 m
#   level 2 (50 m)   -> 6.25 m
#   level 3 (25 m)   -> 3.125 m   <- dune line, ~match for USACE 1 m DEM
#
# nrmax=200 chunks the subgrid build by cell-block; this is the load-bearing
# memory mitigation for the 24 GB RAM budget (build does NOT load the full
# subgrid raster at once).
reclass_table = "/home/zagreus/nj_sandy_sfincs/data/roughness/NLCD_CONUS_mapping.csv"

elevation_list = [
    {"elevation": "usace_nj_2010"},
    {"elevation": "cudem_nj"},
    {"elevation": "nj_10ft_dem", "zmin": 0.001},
    {"elevation": "gebco_nj"},
]
roughness_list = [{"lulc": "nlcd_2012", "reclass_table": reclass_table}]

sf.quadtree_roughness.create(roughness_list=roughness_list, nrmax=200)

sf.quadtree_subgrid.create(
    elevation_list=elevation_list,
    roughness_list=roughness_list,
    nr_subgrid_pixels=8,
    nrmax=2000,  # DO NOT lower this for subgrid! nrcb = nrmax/refi,
    # so smaller nrmax => MANY more blocks => MANY more
    # merge_multi_dataarrays calls + a 132k-iter inner
    # Python loop per block. With nrmax=200, L3 alone is
    # 51M Python iterations + 768 merge calls. nrmax=2000
    # gives ~1 block per level -> 60s total vs 2+ hours.
    # Memory is fine: block = 6 km square @ 3 m subgrid
    # = ~64 MB/layer x 4 layers = ~250 MB working set.
    write_dep_tif=True,
    write_man_tif=True,
)


: 

In [ ]:
# Quick sanity check on the quadtree subgrid table. On quadtree the subgrid
# lives at sf.quadtree_subgrid.data (not sf.subgrid.data) and the variable
# names are z_zmin/z_zmax/uv_havg/uv_navg etc. (per-edge, not per-cell), so
# the regular-grid plot_basemap path doesn't apply. Just sanity-check ranges.
sg = sf.quadtree_subgrid.data
print(f"subgrid variables: {list(sg.data_vars)}")
for v in ["z_zmin", "z_zmax", "uv_havg", "uv_navg"]:
    if v in sg:
        print(
            f"  {v}: shape={sg[v].shape}  "
            f"min={float(sg[v].min()):.3f}  max={float(sg[v].max()):.3f}"
        )


: 

In [ ]:
# --- End of Phase 1: write the static model to disk ---
# Grid, elevation, mask, subgrid tables, and obs points are now on disk.
# None of this depends on forcing, so you don't need to rebuild it when
# iterating on water level / wind / pressure — Phase 2 reopens from here.
sf.write()

# Free the build-time memory. The data-catalog cache + source DEMs held
# ~10 GB resident after the build; Phase 2 opens a fresh handle and doesn't
# need any of it. (Restarting the kernel here works just as well.)
del sf
import gc

gc.collect()

: 

## Phase 2 — Forcing & run

Forcing-only iteration (water level, wind/pressure, rainfall, discharge). Cheap — no DEM or subgrid rebuild. The entry-point cell reopens the static Phase-1 model in `r+` mode and is safe after a kernel restart.

In [ ]:
# --- Phase 2 entry point: reopen the static quadtree model from disk ---
model_root = "../model_quadtree"
data_libs = ["/home/zagreus/nj_sandy_sfincs/data/data_catalog.yml"]

sf = SfincsModel(model_root, data_libs=data_libs, mode="r+")
print(f"reopened {sf.grid_type} model at {model_root}")


: 

### 1. Add water level time-series as forcing

Observed NOAA CO-OPS hourly water levels (NAVD88) at three complete-record gauges (Battery, Atlantic City, Cape May). Sandy Hook (8531680) is **excluded from forcing** — its NaN tail after the mid-storm failure collapses the northern boundary; the Battery anchors that latitude. `merge=False` replaces the stale `r+` forcing; `buffer=100000` reaches Atlantic City so the boundary keeps Sandy's alongshore gradient.

In [ ]:
# Set the simulation period to cover Hurricane Sandy's landfall.
# tstart == tref so the model gets ~24 h of calm tide before Sandy arrives.
sf.config.update(
    {
        "tref": datetime(2012, 10, 28),
        "tstart": datetime(2012, 10, 28),
        "tstop": datetime(2012, 10, 31),
        "tspinup": 3600.0,
        "coriolis": 1,
        "latitude": 40.32,
        "advection": 1,
        "dtmapout": 3600.0,
        "dtmaxout": 86400.0,
        "dthisout": 600.0,
        # --- Phase 3 SnapWave (X1 config) -----------------------------------
        # snapwave=1 turns the incident wave solver on; it consumes the
        # snapwave_mask + the ASCII snapwave.{bnd,bhs,btp,bwd,bds} written
        # below. snapwave_igwaves=1 binds IG waves to the incident spectrum at
        # the wave boundary (Herbers 1994) -- the IG runup is what overtops the
        # highest dunes during Sandy. dtwave=1800 s is the SnapWave coupling
        # interval (matches Leijnse). We deliberately do NOT set the wider
        # physics block (gamma/alpha/niter/dtheta/sector): tested 2026-06-01 it
        # made the surf-zone hm0 spikes WORSE, not better (the spikes were a
        # bathymetry-cliff artifact, since fixed by re-clipping CUDEM).
        "snapwave": 1,
        "snapwave_igwaves": 1,
        "dtwave": 1800.0,
    }
)

# Apply observed NOAA CO-OPS water levels to the boundary cells.
# (Carried over verbatim from the regular-grid notebook -- same gauges,
# same merge=False reason, same 100 km buffer to reach Atlantic City.)
sf.water_level.create(
    geodataset="noaa_sandy_nj",
    buffer=100000,
    merge=False,
)

: 

In [ ]:
# --- X1 SnapWave boundary conditions --------------------------------------
# Inject the ERA5 incident-wave spectrum at the offshore SFINCS waterlevel
# boundary (mask==2, ~-10 m) -- points guaranteed inside the mesh in real
# water. This is the working "X1" setup (2026-05-26) that replaced the earlier
# create_from_grid call, which placed input points at raw ERA5 coords OUTSIDE
# the mesh (depth=0 -> clamp -> IG blow-up -> SFINCS crash).
#
# The forcing is UNIFORM alongshore: the ERA5 0.5 deg grid (~55 km) is far too
# coarse to resolve our ~40 km of coast, and only one offshore node (-74.0,
# 40.0) is valid (the 40.5 nodes are NaN/land). Per-point ERA5 is a future
# improvement (would need a finer wave hindcast).
#
# The ASCII files are written post-write (in the "finalize" cell after
# sf.write()) so they survive the write; here we only compute + store them.
import numpy as np

N_SUPPORT = 7  # support points spread alongshore on the offshore boundary

# 1. Support points = the SEAWARD (Atlantic) edge of the mask==2 boundary.
#    Bin the boundary cells by northing and take the easternmost (max x) cell
#    per bin. This traces the offshore edge and avoids the bay/western boundary
#    cells, where the uniform offshore spectrum would be spurious.
_grid = sf.quadtree_grid.data
_bxy = _grid.grid.face_coordinates[_grid["mask"].values == 2]  # (n, 2) UTM
_ybins = np.linspace(_bxy[:, 1].min(), _bxy[:, 1].max(), N_SUPPORT + 1)
snapwave_pts = np.array(
    [
        grp[np.argmax(grp[:, 0])]
        for k in range(N_SUPPORT)
        for grp in [_bxy[(_bxy[:, 1] >= _ybins[k]) & (_bxy[:, 1] <= _ybins[k + 1])]]
        if len(grp)
    ]
)
print(
    f"snapwave support points: {len(snapwave_pts)} on the seaward mask==2 boundary"
    f"  (x {snapwave_pts[:, 0].min():.0f}..{snapwave_pts[:, 0].max():.0f},"
    f" northing {snapwave_pts[:, 1].min():.0f}..{snapwave_pts[:, 1].max():.0f})"
)

# 2. Uniform forcing from the nearest valid ERA5 node (-74.0, 40.0).
_ew = sf.data_catalog.get_rasterdataset("era5_waves_nj")
_node = _ew.sel(x=-74.0, y=40.0, method="nearest")
snapwave_t = (_node["time"].values - _node["time"].values[0]) / np.timedelta64(1, "s")
snapwave_hs = _node["hs"].values
snapwave_tp = _node["tp"].values
snapwave_wd = _node["wd"].values
snapwave_ds = np.full_like(snapwave_hs, 30.0)  # ERA5 has no dir-spreading; 30 deg
print(
    f"ERA5 node (-74.0, 40.0): hs {snapwave_hs.min():.2f}-{snapwave_hs.max():.2f} m, "
    f"tp {snapwave_tp.min():.1f}-{snapwave_tp.max():.1f} s, {len(snapwave_t)} hourly steps"
)

: 

### Wavemaker line — disabled for X1

The earlier design (following Leijnse et al., Carolinas/Florence 2025) injected the SnapWave radiation-stress forcing into SFINCS along a single alongshore **wavemaker line** at the ≈ −5 m contour. The **X1** setup doesn't use it: with `snapwave_mask == SFINCS mask`, wave setup is handed to SFINCS across the shared active domain directly. The wavemaker remains **Plan B** — revisit if Sandy Hook back-bay validation needs a dedicated bay-side wave source (`scripts/build_wavemaker_line.py` builds the line from the CUDEM −5 m contour).

In [ ]:
# Wavemaker line is NOT used in the X1 setup: with snapwave_mask == SFINCS mask,
# the wave setup is handed across the shared active domain directly. Left
# disabled; see the markdown above (Plan B if back-bay validation needs it).
print("wavemaker: disabled for X1 (shared snapwave/SFINCS mask)")

: 

### 2. Add ERA5 wind + pressure forcing

ERA5 hourly 10-m winds and MSLP. Wind setup on the NJ shelf was Sandy's dominant surge driver here. With observed-gauge boundaries the inverse-barometer signal is already in the BC, so internal MSLP mainly drives the *internal* pressure gradient.

In [ ]:
# ERA5 file is pre-renamed to hydromt_sfincs conventions
# (wind10_u, wind10_v in m/s; press_msl in Pa; coords time/y/x).
# Both methods clip to the model bbox + time range internally.
sf.wind.create(wind="era5_nj")
sf.pressure.create(press="era5_nj")

: 

### 2b. Add rainfall forcing (NOAA AORC)

Direct precipitation from **NOAA AORC v1.1** (~1 km hourly, obs-grounded; covers 2012 unlike MRMS). ~34 mm total over the window — secondary here, since Sandy was surge-dominated and the record rainfall fell inland. `cumulative_input=True` because `APCP_surface` is mm accumulated per 1 h. `aggregate=False` keeps the field spatially distributed (`netamprfile`).

In [ ]:
# AORC precip is accumulated mm per 1-hour interval -> cumulative_input=True.
# aggregate=False keeps it spatially distributed (writes netamprfile / sfincs_netampr.nc).
sf.precipitation.create(precip="aorc_sandy_nj", cumulative_input=True, aggregate=False)

_pr = sf.precipitation.data
_var = "precip_2d" if "precip_2d" in _pr else list(_pr.data_vars)[0]
print(
    f"precip [mm/hr]: peak={float(_pr[_var].max()):.2f}  "
    f"domain-mean={float(_pr[_var].mean()):.3f}  "
    f"grid={dict(_pr[_var].sizes)}"
)

: 

### 2c. Add river discharge (USGS)

Fluvial inflow at the two gauged coastal rivers: **Shark River** (01407705) and **Navesink/Shrewsbury** via Swimming R (01407500). Daily-mean (only resolution archived for these small gauges in 2012), peaks ≈ 3.5 / 7.9 m³/s — minor next to the surge but part of the compound framing. The src points sit at the wet estuary inflow cells, not the upstream gauges. `merge=False` for the usual `r+` reason. Uses `discharge_points.create` (not `rivers.create_river_inflow`), so the mask isn't touched.

In [ ]:
# Daily-mean discharge at 2 domain inflows (Shark River, Navesink). The
# geodataset's point coords place the src cells; merge=False replaces any
# stale src/dis loaded by r+. Writes sfincs.src + sfincs.dis (no mask edit).
sf.discharge_points.create(geodataset="usgs_sandy_discharge", merge=False)

_dis = sf.discharge_points.data
print(
    f"discharge src points: {_dis.sizes.get('index', _dis.sizes.get('stations'))}  "
    f"peak={float(_dis['dis'].max()):.2f} m3/s"
)

: 

### 2d. Add infiltration (NRCS Curve Number)

Without infiltration the ~34 mm of rain ponds as a thin film on every interior cell (≈ 2.9 M m³ above the surge reach — an artifact). **SCS Curve Number** = f(NLCD 2012, SSURGO HSG), built by `scripts/build_cn_nj.py`. SFINCS consumes **rainfall only** with SCS, so coastal inundation is unaffected. `antecedent_moisture=None` → CN II (average). Placed in Phase 2 only to skip a costly Phase-1 subgrid rebuild.

In [ ]:
# SCS Curve Number infiltration on the quadtree mesh.
#
# Uses sf.quadtree_infiltration (not sf.infiltration — that's the regular-grid
# API and errors on quadtree models). antecedent_moisture=None reads the 'cn'
# variable (CN II / average) directly. The scs variable lives on the quadtree
# grid; SCS only consumes rainfall, never surge.
#
# KNOWN HYDROMT BUG: the quadtree component sets sfincs.inp keys
# `infiltration_file = infiltration.nc` + `infiltration_type = cna` but its
# write() method is a `pass` — no infiltration.nc is ever written. SFINCS then
# errors at runtime ("Infiltration netcdf file not found"). The post-write
# patch cell below strips the orphan config lines so SFINCS runs without
# infiltration. Effect on Sandy validation is small (~+/-0.02 CSI) because rain
# was modest (34 mm) and surge-dominated.
sf.quadtree_infiltration.create_cn(cn="cn_nj", antecedent_moisture=None, nrmax=2000)

_scs = sf.quadtree_grid.data["scs"]
print(
    f"SCS max soil-moisture retention S [inch]: "
    f"mean={float(_scs.where(_scs > 0).mean()):.2f}  max={float(_scs.max()):.2f}  "
    f"(higher S = more infiltration capacity; S=0 over water/impervious)"
)


: 

### 3. Save out model

Writes the forcing (and updated config) into the model directory alongside the Phase 1 static files.

In [ ]:
# Write all model files to disk, then finalize sfincs.inp on disk.
#
# hydromt-sfincs v2.0.0rc2 silently drops some config keys on write (confirmed
# for `latitude`; we treat the SnapWave keys defensively the same way), and the
# infiltration component sets a key without writing its file. We patch the inp
# directly here. Every step is idempotent — remove once the upstream fixes land.
import numpy as np
from pathlib import Path

sf.write()

inp = Path(model_root) / "sfincs.inp"
root = Path(model_root)
text = inp.read_text()

# (a) latitude — without it SFINCS disables Coriolis ("Coriolis: no") even with
#     coriolis=1, because in-memory latitude reverts to 0.0 after write.
LATITUDE_DEG = 40.32  # keep in sync with the config.update cell above
if "\nlatitude" not in text:
    text = text.replace(
        "coriolis             = 1",
        f"coriolis             = 1\nlatitude             = {LATITUDE_DEG}",
    )
    print(f"added latitude = {LATITUDE_DEG}")

# (b) strip orphan infiltration keys: quadtree_infiltration.create_cn sets
#     infiltration_file but its write() is a no-op, so SFINCS aborts
#     "Infiltration netcdf file not found". (Sandy was surge-dominated; dropping
#     infiltration barely moves the validation.)
kept = []
for line in text.splitlines():
    if line.strip().startswith(("infiltration_file", "infiltration_type", "scsfile")):
        print(f"stripped: {line!r}")
        continue
    kept.append(line)
text = "\n".join(kept) + "\n"

# (c) ensure the X1 SnapWave keys are present (same dropped-key class as above).
snapwave_keys = {
    "snapwave": "1",
    "snapwave_igwaves": "1",
    "dtwave": "1800.0",
    "snapwave_bndfile": "snapwave.bnd",
    "snapwave_bhsfile": "snapwave.bhs",
    "snapwave_btpfile": "snapwave.btp",
    "snapwave_bwdfile": "snapwave.bwd",
    "snapwave_bdsfile": "snapwave.bds",
}
present = {ln.split("=")[0].strip() for ln in text.splitlines() if "=" in ln}
for k, v in snapwave_keys.items():
    if k not in present:
        text += f"{k:<20} = {v}\n"
        print(f"added snapwave key: {k} = {v}")
inp.write_text(text)
print(f"patched {inp}")

# (d) write the X1 SnapWave ASCII forcing (uniform across support points),
#     post-write so it survives sf.write(). Remove stale caches first:
#     snapwave.upw (upwind connectivity, keyed to dtheta/sector -> "End of file"
#     crash if mismatched) and snapwave.nc (old create_from_grid path).
for _stale in ("snapwave.upw", "snapwave.nc"):
    (root / _stale).unlink(missing_ok=True)
np.savetxt(root / "snapwave.bnd", snapwave_pts, fmt="%.3f")


def _write_sw(fn, series):
    block = np.tile(np.asarray(series)[:, None], (1, len(snapwave_pts)))
    np.savetxt(
        root / fn,
        np.column_stack([snapwave_t, block]),
        fmt=["%11.1f"] + ["%11.3f"] * len(snapwave_pts),
    )


for _fn, _series in [
    ("snapwave.bhs", snapwave_hs),
    ("snapwave.btp", snapwave_tp),
    ("snapwave.bwd", snapwave_wd),
    ("snapwave.bds", snapwave_ds),
]:
    _write_sw(_fn, _series)
print(f"wrote snapwave.{{bnd,bhs,btp,bwd,bds}} ({len(snapwave_pts)} support points)")

: 

### 4. Run SFINCS via Docker

Runs the model in the official `deltares/sfincs-cpu:latest` container, mounting the model directory at `/data`. The first `docker run` is a small cleanup step — Docker writes output files as root, and SFINCS raises "NetCDF: Not a valid ID" if it can't overwrite a root-owned `sfincs_map.nc` from a previous run. The second `docker run` is the actual simulation.

In [ ]:
log_path = Path(model_root) / "sfincs_log.txt"
model_abs = Path(model_root).resolve()

# Remove stale output files before running — Docker writes them as root, and SFINCS
# gets "NetCDF: Not a valid ID" errors if it tries to overwrite root-owned files.
subprocess.run(
    [
        "docker",
        "run",
        "--rm",
        "-v",
        f"{model_abs}:/data",
        "--entrypoint",
        "/bin/sh",
        "deltares/sfincs-cpu:latest",
        "-c",
        "rm -f /data/sfincs_map.nc /data/sfincs_his.nc",
    ],
    capture_output=True,
)

print(f"Running SFINCS via Docker on {model_abs} ...")
with open(log_path, "w") as log_file:
    result = subprocess.run(
        [
            "docker",
            "run",
            "--rm",
            "-v",
            f"{model_abs}:/data",
            "deltares/sfincs-cpu:latest",
        ],
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )
print(f"Done (return code {result.returncode})")


: 

In [ ]:
# Inspect the run: SFINCS log + output files.
log_path = Path(model_root) / "sfincs_log.txt"
if log_path.exists():
    print(log_path.read_text())
else:
    print("No log file found — SFINCS has not run yet.")

for f in ("sfincs_map.nc", "sfincs_his.nc"):
    print(f"{f} exists: {(Path(model_root) / f).exists()}")

: 

## Phase 3 — Visualization

Read the SFINCS outputs (`sfincs_map.nc`, `sfincs_his.nc`) and inspect the results: peak water levels at obs points, time series, zone stats, validation against the Sandy Hook gauge, and a downscaled flood map. This phase is independent of Phase 1 / Phase 2 — it opens the model fresh in read-only mode, so it works after a kernel restart without re-running the build.

### 1. Read Model Results

In [ ]:
# Open the model read-only for result inspection. Standalone entry point.
model_root = "../model_quadtree"
data_libs = ["/home/zagreus/nj_sandy_sfincs/data/data_catalog.yml"]
model_abs = Path(model_root).resolve()

mod = SfincsModel(model_root, data_libs=data_libs, mode="r")
mod.output.read()

print(f"grid_type: {mod.grid_type}")
print("Output variables available:")
list(mod.output.data.keys())


: 

In [ ]:
# Plot the model layout — grid, boundary cells, and observation points
fig, ax = mod.plot_basemap(fn_out=None, bmap="sat", figsize=(9, 7), geom_names=["obs"])

: 

### Validation: modeled vs observed water level at Sandy Hook

The Sandy Hook gauge (8531680) is the one in-domain temporal record — but it failed at 10-29 23:00, *before* Sandy's true peak. Compare the modeled curve to the gauge **only over the overlap window**; the modeled full-run peak (~3.9 m) is what's comparable to the canonical Sandy Hook storm-tide estimate (~3.86 m).

In [ ]:
# Modeled zs at the sandy_hook_gauge obs point vs the observed NOAA record.
# Model output at obs points (relocated from the removed peak-table cell).
point_zs = mod.output.data["point_zs"]  # (time, station)
point_zb = mod.output.data["point_zb"]  # (station,)
names = [
    n.decode() if isinstance(n, bytes) else str(n)
    for n in point_zs["station_name"].values
]

val = xr.open_dataset(
    "/home/zagreus/nj_sandy_sfincs/data/gtsm/noaa_sandy_validation.nc"
)
obs_sh = val["waterlevel"].sel(stations=8531680)

# locate the sandy_hook_gauge obs point (names may carry trailing padding)
i_sh = next(k for k, n in enumerate(names) if "sandy_hook" in n)
mod_sh = point_zs.isel(stations=i_sh)
zb_sh = float(point_zb.isel(stations=i_sh).values)
mod_sh_wet = mod_sh.where(mod_sh - zb_sh > 0.01)  # only where the cell is wet

# peak comparison over the window the gauge actually covers
gauge_end = pd.Timestamp("2012-10-29 23:00")
mod_overlap = mod_sh.sel(time=slice(None, gauge_end))
print(f"observed peak (pre-failure): {float(obs_sh.max()):.2f} m NAVD88")
print(f"modeled  peak (same window): {float(mod_overlap.max()):.2f} m NAVD88")
print(f"modeled  peak (full run):    {float(mod_sh.max()):.2f} m NAVD88")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(mod_sh["time"], mod_sh_wet.values, lw=2, label="modeled zs (SFINCS)")
ax.plot(obs_sh["time"], obs_sh.values, "k.-", ms=4, label="observed (NOAA 8531680)")
ax.axvline(gauge_end, color="red", ls=":", alpha=0.6, label="gauge fails (10-29 23:00)")
ax.set_ylabel("Water surface elevation [m NAVD88]")
ax.set_xlabel("Time [UTC]")
ax.set_title("Sandy Hook: modeled vs observed water level")
ax.legend()
ax.grid(alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()

: 

### Diagnostic: is the bias the boundary, or the wave setup?

Subtracting the reconstructed Stockdon setup isolates the still-water solution. If the green (no-setup) curve tracks the observed gauge while the blue (with-setup) rides high, the bias is the parametric *setup*, not the boundary. Sign: model − observed.

In [ ]:
# Diagnostic: how much of the Sandy Hook bias is the parametric wave setup?
# We reconstruct the Stockdon setup near the gauge (same ERA5 field + β_f the
# forcing cell 1b uses) and subtract it from the modeled series. FIRST-ORDER
# estimate: it removes the setup imposed at the boundary, but the bay's response
# to a boundary bump is slightly attenuated, so the green curve mildly OVER-
# subtracts. Read green as ~"the still-water solution the gauge actually sees."
# NOTE: reflects the ERA5 forcing — valid after a Phase-2 re-run with cell 1b.
import numpy as np

BETA_F = 0.05
GRAVITY = 9.81
SH_LON, SH_LAT = -74.0091, 40.4669  # NOAA 8531680 gauge location

waves = xr.open_dataset("/home/zagreus/nj_sandy_sfincs/data/waves/era5_waves_nj.nc")
hs_all, tp_all = waves["hs"], waves["tp"]
vmask = np.isfinite(hs_all).any("time").values
lon2d, lat2d = np.meshgrid(hs_all["x"].values, hs_all["y"].values)
iy, ix = np.where(vmask)
vlon, vlat = lon2d[vmask], lat2d[vmask]
k = int(np.argmin((vlon - SH_LON) ** 2 + (vlat - SH_LAT) ** 2))
hs, tp = (
    hs_all.isel(y=int(iy[k]), x=int(ix[k])),
    tp_all.isel(y=int(iy[k]), x=int(ix[k])),
)
L0 = GRAVITY * tp**2 / (2.0 * np.pi)
eta = (0.35 * BETA_F * np.sqrt(hs * L0)).fillna(0.0)
eta_m = eta.interp(time=mod_sh["time"], kwargs={"fill_value": 0.0})

mod_nosetup = mod_sh - eta_m  # approx no-wave still-water baseline at the gauge

# residuals vs the observed record over the window the gauge covers
obs_t = pd.to_datetime(obs_sh["time"].values)
mod_t = pd.to_datetime(mod_sh["time"].values)
obs_i = np.interp(
    mod_t.view("i8"), obs_t.view("i8"), obs_sh.values, left=np.nan, right=np.nan
)
ov = mod_t <= gauge_end
print(f"peak setup reconstructed near gauge: {float(eta_m.max()):.2f} m")
print(
    f"mean residual (to gauge fail)  WITH setup: {np.nanmean(mod_sh.values[ov] - obs_i[ov]):+.2f} m"
)
print(
    f"mean residual (to gauge fail)   NO  setup: {np.nanmean(mod_nosetup.values[ov] - obs_i[ov]):+.2f} m"
)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(
    mod_sh["time"],
    mod_sh.values,
    lw=2,
    color="tab:blue",
    label="modeled (with Stockdon setup)",
)
ax.plot(
    mod_sh["time"],
    mod_nosetup.values,
    lw=1.8,
    ls="--",
    color="tab:green",
    label="modeled − setup (≈ no-wave baseline)",
)
ax.plot(obs_sh["time"], obs_sh.values, "k.-", ms=4, label="observed (NOAA 8531680)")
ax.axvline(gauge_end, color="red", ls=":", alpha=0.6, label="gauge fails")
ax.set_ylabel("Water surface elevation [m NAVD88]")
ax.set_xlabel("Time [UTC]")
ax.set_title("Sandy Hook: how much of the high bias is the wave setup?")
ax.legend()
ax.grid(alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()


: 

### Validation: pre-storm tides at the two in-domain USGS gauges

Two USGS estuary gauges in NAVD88 — Shark River at Belmar (south) and Shrewsbury at Sea Bright (mid-north back-bay). Both records stop ~10-29 04:00 UTC, before the peak (all in-domain permanent gauges failed mid-storm), so this is a **tidal check** (range / phase), sampled at the nearest grid cell to each gauge.

In [ ]:
# Pre-storm tidal validation at the two in-domain USGS gauges (NAVD88).
# Quadtree fix: the map output has dims (time, nmesh2d_face) with no xc/yc grid
# coords, so we sample the nearest WET mesh face (via the grid face centroids)
# instead of the old regular-grid xc/yc + isel(y=, x=).
import numpy as np
from pyproj import Transformer

usgs = xr.open_dataset("/home/zagreus/nj_sandy_sfincs/data/gtsm/usgs_sandy_tidal_nj.nc")
zs_face = mod.output.data["zs"]                    # (time, nmesh2d_face)
fc = mod.quadtree_grid.data.grid.face_coordinates  # (nmesh2d_face, 2) in mod.crs
wet_any = np.isfinite(zs_face.values).any(axis=0)  # faces wet at some timestep
to_utm = Transformer.from_crs(4326, mod.crs, always_xy=True)
gauge_label = {
    1407770: "Shark River @ Belmar (south)",
    1407600: "Shrewsbury @ Sea Bright (back-bay)",
}


def nearest_wet_face(X, Y):
    d2 = np.where(wet_any, (fc[:, 0] - X) ** 2 + (fc[:, 1] - Y) ** 2, np.inf)
    j = int(np.argmin(d2))
    return j, float(np.hypot(fc[j, 0] - X, fc[j, 1] - Y))


fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
for ax, sid in zip(axes, usgs["stations"].values):
    o = usgs["waterlevel"].sel(stations=sid).dropna("time")
    X, Y = to_utm.transform(
        float(usgs["lon"].sel(stations=sid)), float(usgs["lat"].sel(stations=sid))
    )
    idx, dist = nearest_wet_face(X, Y)
    m = zs_face.isel(nmesh2d_face=idx)
    mo = m.sel(time=slice(o["time"].min(), o["time"].max()))  # overlap window
    rng_o = float(o.max() - o.min())
    rng_m = float(mo.max() - mo.min())
    print(
        f"{gauge_label[int(sid)]:34s}: tidal range obs {rng_o:.2f} m vs modeled "
        f"{rng_m:.2f} m  (nearest wet face {dist:.0f} m away)"
    )
    ax.plot(m["time"], m.values, lw=2, color="tab:blue", label="modeled zs (nearest wet face)")
    ax.plot(o["time"], o.values, "k.-", ms=3, label="observed USGS")
    ax.set_title(f"{gauge_label[int(sid)]}  —  pre-storm tide (record ends ~10-29 04:00 UTC)")
    ax.set_ylabel("WSE [m NAVD88]")
    ax.legend(loc="upper left")
    ax.grid(alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()

: 

### Validation: the one gauge that caught the peak — USGS storm-tide sensor

A USGS rapid-deployment storm-tide sensor on the open coast at Monmouth Beach (40.37°N, two co-located units), via the STN Flood Event Viewer (NAVD88, GMT). These are **wave sensors**: raw signal includes wave oscillations, and being mounted ~9 ft NAVD88 they de-water in troughs (we mask the floored samples, so the 30-min mean is valid only near the submerged peak). The two units disagree by ~1.5 m — read as a *bracket*, not a precise number.

In [ ]:
# Open-coast storm-tide sensor (peak-capturing) vs modeled zs.
# Quadtree fix: sample the nearest mesh face (no xc/yc grid coords).
import numpy as np
from pyproj import Transformer

sst = xr.open_dataset("/home/zagreus/nj_sandy_sfincs/data/gtsm/sandy_storm_tide_nj.nc")
zs_face = mod.output.data["zs"]                    # (time, nmesh2d_face)
fc = mod.quadtree_grid.data.grid.face_coordinates  # (nmesh2d_face, 2) in mod.crs
to_utm = Transformer.from_crs(4326, mod.crs, always_xy=True)
X, Y = to_utm.transform(
    float(sst["lon"].isel(stations=0)), float(sst["lat"].isel(stations=0))
)
idx = int(np.argmin((fc[:, 0] - X) ** 2 + (fc[:, 1] - Y) ** 2))
m = zs_face.isel(nmesh2d_face=idx)

print(f"modeled open-coast peak still-water (zs): {float(m.max()):.2f} m NAVD88")
for sid in sst["stations"].values:
    st = sst["stormtide_m"].sel(stations=sid)
    wm = sst["wavemax_m"].sel(stations=sid)
    print(
        f"  sensor {int(sid)}: storm-tide (still-water) peak {float(st.max()):.2f} m | "
        f"wave-crest peak {float(wm.max()):.2f} m"
    )

fig, ax = plt.subplots(figsize=(11, 5))
wm0 = sst["wavemax_m"].sel(stations=int(sst["stations"][-1])).dropna("time")
ax.fill_between(
    wm0["time"].values, 0, wm0.values, color="tab:orange", alpha=0.12,
    label="sensor wave-crest envelope (incl. waves)",
)
ax.plot(m["time"], m.values, lw=2.2, color="tab:blue", label="modeled zs (nearest face)")
for sid, c in zip(sst["stations"].values, ["k", "dimgray"]):
    st = sst["stormtide_m"].sel(stations=sid).dropna("time")
    ax.plot(st["time"], st.values, ".-", ms=3, color=c,
            label=f"sensor {int(sid)} storm-tide (30-min mean, wet only)")
ax.set_xlim(np.datetime64("2012-10-29T18"), np.datetime64("2012-10-30T12"))
ax.set_ylabel("Water surface elevation [m NAVD88]")
ax.set_xlabel("Time [UTC]")
ax.set_title("Open-coast storm-tide sensor (40.37N) vs modeled - Sandy peak")
ax.legend(loc="upper right", fontsize=8)
ax.grid(alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()

# Takeaway: the two co-located wave sensors bracket the model's still-water peak,
# so they don't cleanly resolve the open-coast over/under here - but their wave
# crests (~5-6 m) match the highest HWMs, confirming that tail is wave runup the
# still-water model omits. The HWMs remain the cleaner spatial validation; this is
# the only in-domain record that survived to the peak.

: 

### 2. Downscale Flood Map

Built from the finest subgrid DEM (`dep_subgrid_lev3.tif`, ~3 m), which covers the dune/surf/back-bay-shore corridor (`surf_dune` refinement, z∈[−8,+3] m) where the HWMs and the MOTF land cells live.

> **Why L3-only (mosaic tested & rejected, 2026-06-02):** a finest-wins L1+L2+L3 mosaic adds **0.00 km²** of new MOTF-wet *land* — the L2/L1 fill is deep estuary water (z<−8 m) that this map masks out anyway. The 15.9 km² “model-dry” back-bay miss below is **already inside the L3 footprint**, so it is genuine under-prediction (→ boundary/inlet work), not a flood-map coverage gap.

In [ ]:
# Reuse `mod` opened above — no need to open a second SfincsModel
da_zsmax = mod.output.data["zsmax"].max(dim="timemax")

# Quadtree subgrid writes one DEM per refinement level; the dune/surf flooding +
# the HWMs live in the finest (L3, ~3 m). Downscaling the full 69M-px L3 DEM in
# one shot OOMs (>24 GB): on a ROTATED grid, downscale_floodmap converts the
# whole DEM to an unstructured mesh. Passing dep as a PATH + floodmap_fn makes it
# tile the DEM into nrmax-sized blocks (bounded memory) and write to disk; we
# then read the result back. nrmax=1000 -> ~1M px/block.
depfile = str(model_abs / "subgrid" / "dep_subgrid_lev3.tif")
floodmap_fn = str(model_abs / "floodmap_hmax_lev3.tif")
utils.downscale_floodmap(
    zsmax=da_zsmax, dep=depfile, hmin=0.05, floodmap_fn=floodmap_fn, nrmax=1000
)

# Read back the downscaled flood depth + the matching L3 DEM (for the figure and
# the HWM sampling below). Use positional (.values) masking to avoid float-coord
# alignment issues between the two rasters.
import rioxarray

# Read both at FULL native resolution. (mod.data_catalog.get_rasterdataset
# can return different overview levels for the two COGs -> a shape mismatch
# on the .where below, e.g. 12488x5512 vs 6244x2756.)
da_hmax = rioxarray.open_rasterio(floodmap_fn, masked=True).squeeze(drop=True)
da_dep = rioxarray.open_rasterio(depfile, masked=True).squeeze(drop=True)

# The quadtree grid is ROTATED, and the tiled downscale landed the floodmap on
# the DEM's half-res overview. De-rotate the floodmap to a north-up grid and
# match the DEM onto it. This (a) puts both on one identical grid (so the
# .where and the later da_dep + da_hmax align) and (b) makes the downstream
# raster row/col sampling (HWMs in the next cells, MOTF) — which assumes an
# axis-aligned transform (X-T.c)/T.a — actually correct.
da_hmax = da_hmax.rio.reproject(da_hmax.rio.crs)
da_dep = da_dep.rio.reproject_match(da_hmax)
da_hmax = da_hmax.where(da_dep.values > -0.5)  # drop deep ocean from colour scale
da_hmax.name = "hmax"  # plot_basemap/to_dataset needs a named DataArray
print(f"flood map (L3): {tuple(da_hmax.shape)}, res {da_hmax.rio.resolution()[0]:.1f} m")

: 

In [ ]:
fig, ax = mod.plot_basemap(
    fn_out=None,
    figsize=(8, 6),
    variable=da_hmax,
    plot_bounds=False,
    plot_geoms=False,
    bmap="sat",
    zoomlevel=11,
    vmin=0,
    vmax=5.0,
    cbar_kwargs={"shrink": 0.6, "anchor": (0, 0)},
)
ax.set_title(f"SFINCS maximum water depth")

: 

### Validation: USGS High Water Marks (spatial — "more or less than Sandy?")

**31 USGS HWMs** in-domain (peak NAVD88 elevations from the rapid-response survey, STN event 24, downloaded by `scripts/download_sandy_hwms.py`). Each mark is compared to the wettest **genuinely-flooded** model cell within a 50 m radius:

- **50 m radius** — the nearest 6 m pixel often lands on the building/raised lot the mark sits on (reads dry), so we look at the adjacent flooded street/yard.
- **`DEPTH_MIN`** — required, else a thin rain film on high ground gives a spuriously high WSE.
- **`GROUND_CAP`** — also require `dep ≤ obs + 0.5 m` so the search can't grab a dune/structure cell whose own water surface is well above the mark.

Sign: + = model higher. Headline = quality ≤ 2 subset.

In [ ]:
# Compare modeled peak STILL-WATER level vs USGS Sandy High Water Marks.
import numpy as np

DEPTH_MIN = 0.15  # m; only compare to genuinely flooded cells, not thin rain film.
GROUND_CAP = 0.5  # m; a mark at elevation `obs` can only have been wet by water
# from cells whose GROUND sits at/below obs (+tol). Without this
# the 50 m search can grab a higher dune/structure cell whose own
# water surface is well above the mark -> a spurious over-predict
# (it was turning one q3 mark into a +2 m outlier).

# Reuse the tiled flood map from the downscale cell above (re-running
# downscale_floodmap here on the in-memory DataArray would OOM on the
# rotated grid — see that cell).
hmax_val = da_hmax
hwm = gpd.read_file("/home/zagreus/nj_sandy_sfincs/data/validation/sandy_hwms.geojson")
hwm = hwm.to_crs(da_dep.rio.crs)

depth, dep_arr, wse = hmax_val.values, da_dep.values, (da_dep + hmax_val).values
if depth.ndim == 3:
    depth, wse, dep_arr = depth[0], wse[0], dep_arr[0]
T = da_dep.rio.transform()
ny, nx = wse.shape
rad = int(round(50 / abs(T.a)))

obs = hwm["elev_m"].values
qual = hwm["quality"].values.astype(float)
mod_wse = np.full(len(obs), np.nan)
for k, (X, Y) in enumerate(zip(hwm.geometry.x.values, hwm.geometry.y.values)):
    col, row = int((X - T.c) / T.a), int((Y - T.f) / T.e)
    if 0 <= row < ny and 0 <= col < nx:
        sl = (
            slice(max(0, row - rad), row + rad + 1),
            slice(max(0, col - rad), col + rad + 1),
        )
        ws, hh, dd = wse[sl], depth[sl], dep_arr[sl]
        flooded = (hh >= DEPTH_MIN) & (dd <= obs[k] + GROUND_CAP)
        if flooded.any():
            mod_wse[k] = np.nanmax(np.where(flooded, ws, np.nan))

wet = np.isfinite(mod_wse)
resid = mod_wse - obs  # + = model higher than observed
q2 = qual <= 2


def report(label, m):
    if m.sum() == 0:
        print(f"{label}: (none)")
        return
    r = resid[m]
    print(
        f"{label}: n={int(m.sum()):2d}  mean={r.mean():+.2f}  median={np.median(r):+.2f}  "
        f"RMSE={np.sqrt((r**2).mean()):.2f}  within±0.5 m={np.mean(np.abs(r) < 0.5) * 100:.0f}%"
    )


print(
    f"model flooded within 50 m of {wet.sum()}/{len(obs)} HWMs  (+ = model over-predicts)"
)
report("HEADLINE  quality<=2 ", wet & q2)
report("          quality<=3 ", wet & (qual <= 3))
report("          all wet     ", wet)
print(
    f"model DRY at {int((~wet).sum())} HWMs — still-water can't reach them (runup candidates)"
)

# Scatter — emphasise the trustworthy q<=2 subset; q3-4 shown hollow.
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(
    obs[wet & ~q2],
    mod_wse[wet & ~q2],
    facecolor="none",
    edgecolor="grey",
    s=55,
    linewidth=0.8,
    label="q3-4 (low survey quality)",
)
sc = ax.scatter(
    obs[wet & q2],
    mod_wse[wet & q2],
    c=qual[wet & q2],
    cmap="viridis_r",
    s=60,
    edgecolor="k",
    linewidth=0.4,
    vmin=1,
    vmax=5,
    label="q1-2 (headline)",
)
lim = [1.8, 6.0]
ax.plot(lim, lim, "k--", lw=1, label="1:1")
ax.fill_between(
    lim, [l - 0.5 for l in lim], [l + 0.5 for l in lim], color="grey", alpha=0.15
)
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_aspect("equal")
ax.set_xlabel("Observed HWM [m NAVD88]")
ax.set_ylabel("Modeled still-water WSE [m NAVD88]")
ax.set_title("Modeled still-water vs USGS HWMs")
fig.colorbar(sc, ax=ax, shrink=0.8, label="HWM quality (1=best)")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()

: 

### Validation map: where does the model over/under-predict?

The 31 HWMs on the flood map, colored by residual (model − obs). Red = model higher, blue = model lower, black × = dry in the model. Useful for spotting whether the misses cluster spatially (e.g. behind a back-bay shore) or scatter randomly.

In [ ]:
# HWM residuals on the flood map (reuses hwm, resid, wet from the cell above).
import numpy as np

fig, ax = mod.plot_basemap(
    fn_out=None,
    figsize=(9, 7),
    variable=da_hmax,
    plot_bounds=False,
    plot_geoms=False,
    bmap="sat",
    zoomlevel=11,
    vmin=0,
    vmax=5,
    cmap="Blues",
    cbar_kwargs={"shrink": 0.5, "label": "Modeled depth [m]"},
)
hx, hy = hwm.geometry.x.values, hwm.geometry.y.values
sc = ax.scatter(
    hx[wet],
    hy[wet],
    c=resid[wet],
    cmap="RdBu_r",
    vmin=-1.5,
    vmax=1.5,
    s=70,
    edgecolor="k",
    linewidth=0.6,
    zorder=5,
)
ax.scatter(
    hx[~wet],
    hy[~wet],
    marker="x",
    color="k",
    s=70,
    linewidth=1.6,
    zorder=6,
    label=f"model dry ({int((~wet).sum())})",
)
fig.colorbar(sc, ax=ax, shrink=0.5, label="HWM residual: model − obs [m]")
ax.legend(loc="upper right")
ax.set_title("Sandy HWM residuals (red = model over-predicts, blue = under)")
plt.tight_layout()

: 

### Validation: spatial extent vs FEMA MOTF Sandy surge footprint

Footprint-level check: does the model flood the same *area* the FEMA MOTF Sandy extent does? We rasterise MOTF to the model subgrid, restrict to land cells, and score the standard inundation metrics — **CSI** (hits / (hits + miss + FA)), **POD**, **FAR** — plus a categorical hit / miss / false-alarm map.

> **Caveat:** MOTF is a static, HWM/sensor-interpolated "bathtub" surface — not a hydrodynamic run — and shares provenance with our HWMs. Treat this as an extent **consistency** check, not independent validation.

In [ ]:
# Spatial-extent validation vs FEMA MOTF Sandy surge footprint.
# Both rasters are EPSG:32618, so we sample model cells at MOTF pixel centers in
# pure numpy — no GDAL warping needed (avoids known reproject crashes in this env).
import numpy as np
import rasterio
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

DEPTH_MIN = 0.15  # m; match the HWM cell's wet threshold

with rasterio.open(
    "/home/zagreus/nj_sandy_sfincs/data/validation/sandy_motf_extent.tif"
) as r:
    motf = r.read(1)
    mtf = r.transform
    m_nd = r.nodata
mod_t = da_dep.rio.transform()
mh, mw = motf.shape

# Map MOTF pixel centers -> nearest model subgrid pixel index (both UTM 18N).
Xc = mtf.c + (np.arange(mw) + 0.5) * mtf.a
Yc = mtf.f + (np.arange(mh) + 0.5) * mtf.e
mc = np.clip(((Xc - mod_t.c) / mod_t.a).astype(int), 0, da_dep.shape[-1] - 1)
mr = np.clip(((Yc - mod_t.f) / mod_t.e).astype(int), 0, da_dep.shape[-2] - 1)
rr, cc = np.meshgrid(mr, mc, indexing="ij")


def _2d(a):
    return a[0] if a.ndim == 3 else a


dep_at = _2d(da_dep.values)[rr, cc]
h_at = _2d(da_hmax.values)[rr, cc]

motf_wet = motf == 1
mod_wet = (h_at >= DEPTH_MIN) & np.isfinite(h_at)
land_in = (motf != m_nd) & (dep_at > 0.0)  # land cells inside the domain
hits = motf_wet & mod_wet & land_in
miss = motf_wet & ~mod_wet & land_in
fa = ~motf_wet & mod_wet & land_in
nh, nm, nf = int(hits.sum()), int(miss.sum()), int(fa.sum())
PIX = mtf.a * abs(mtf.e) / 1e6  # km^2 per pixel
CSI = nh / (nh + nm + nf)
POD = nh / (nh + nm) if (nh + nm) else 0.0
FAR = nf / (nh + nf) if (nh + nf) else 0.0
motf_land = (motf_wet & land_in).sum() * PIX
mod_land = (mod_wet & land_in).sum() * PIX
print(f"MOTF flooded land in domain : {motf_land:.1f} km2")
print(f"Model wet land in domain    : {mod_land:.1f} km2")
print(f"  hits {nh * PIX:5.1f}  miss {nm * PIX:5.1f}  false-alarm {nf * PIX:5.1f}  km2")
print(f"  CSI={CSI:.2f}   POD (hit rate)={POD:.2f}   FAR={FAR:.2f}")

# Categorical difference map: hits / misses / false alarms.
cat = np.zeros_like(motf, dtype="uint8")
cat[hits] = 1
cat[miss] = 2
cat[fa] = 3
cmap = ListedColormap(
    [
        (1, 1, 1, 0),
        (0.20, 0.60, 0.30, 1.0),
        (0.20, 0.40, 0.85, 1.0),
        (0.85, 0.20, 0.20, 1.0),
    ]
)
ext = [mtf.c, mtf.c + mw * mtf.a, mtf.f + mh * mtf.e, mtf.f]
mod_ext = [
    mod_t.c,
    mod_t.c + da_dep.shape[-1] * mod_t.a,
    mod_t.f + da_dep.shape[-2] * mod_t.e,
    mod_t.f,
]

fig, ax = plt.subplots(figsize=(7.5, 9))
ax.set_facecolor("#f4f4f0")
ax.imshow(
    _2d(da_dep.values),
    extent=mod_ext,
    cmap="Greys",
    vmin=-5,
    vmax=20,
    alpha=0.45,
    origin="upper",
    interpolation="nearest",
)
ax.imshow(
    cat, cmap=cmap, vmin=0, vmax=3, extent=ext, origin="upper", interpolation="nearest"
)
ax.set_aspect("equal")
ax.set_xlim(ext[0], ext[1])
ax.set_ylim(ext[2], ext[3])
ax.set_xlabel("Easting [m, UTM 18N]")
ax.set_ylabel("Northing [m]")
ax.legend(
    handles=[
        Patch(color=cmap(1), label=f"hit ({nh * PIX:.1f} km²)"),
        Patch(color=cmap(2), label=f"miss — MOTF wet, model dry ({nm * PIX:.1f} km²)"),
        Patch(
            color=cmap(3),
            label=f"false alarm — model wet, MOTF dry ({nf * PIX:.1f} km²)",
        ),
    ],
    loc="upper right",
    fontsize=8,
    framealpha=0.95,
)
ax.set_title(
    f"Modeled flood vs FEMA MOTF Sandy extent  —  "
    f"CSI={CSI:.2f}   POD={POD:.2f}   FAR={FAR:.2f}"
)
plt.tight_layout()

# Takeaway (this run): misses cluster in the back-bays / inland lows (runup + 50 m
# connectivity limits, same family as Deal Lake & Shrewsbury); false alarms at
# Sandy Hook spit & a few southern spots (the bay setup leakage we saw at the
# gauges). Extent picture matches the HWM residual map.

: 

In [ ]:
# Reload obs points from the written model (gis/obs.geojson) rather than
# relying on `obs_all` from the Phase 1 build cell — keeps the Visualization
# section runnable standalone after a kernel restart.
obs_all = gpd.read_file(model_abs / "gis" / "obs.geojson")

flood_wgs84 = da_hmax.where(da_hmax > 0.05).rio.reproject(
    "EPSG:4326", nodata=float("nan")
)

flood_map = flood_wgs84.hvplot.image(
    x="x",
    y="y",
    cmap="viridis",
    clim=(0, 5),
    geo=True,
    tiles="EsriImagery",
    alpha=0.75,
    width=900,
    height=700,
    title="Max flood depth — Sandy (NOAA gauges + ERA5 + Stockdon setup + AORC rain + USGS discharge)",
    clabel="Flood depth [m]",
)

obs_layer = obs_all.hvplot.points(
    geo=True,
    color="red",
    size=60,
    hover_cols=["name"],
)

flood_map * obs_layer

: 